In [1]:
import numpy as np
import pandas as pd
import spacy
import codecs, sys
import random
from collections import Counter
import pickle
from nltk.tokenize import word_tokenize


In [ ]:
import pandas as pd
import json

class StopwordRemover:
    def __init__(self, haseeb_json=None, chtgpt_csv=None):
        """
        Initialize the StopwordRemover class with optional stopwords sources.
        
        Args:
            haseeb_json (str): JSON string containing Haseeb Elahi's stopwords.
            chtgpt_csv (str): Path to the CSV file containing Chtgpt's stopwords.
        """
        self.haseeb_stopwords = set(self._load_haseeb_stopwords(haseeb_json)) if haseeb_json else None
        self.chtgpt_stopwords = set(self._load_chtgpt_stopwords(chtgpt_csv)) if chtgpt_csv else None

    def _load_haseeb_stopwords(self, json_string):
        """
        Load stopwords from Haseeb Elahi's JSON string.
        
        Args:
            json_string (str): JSON string containing stopwords.
        
        Returns:
            list: List of stopwords.
        """
        stop_words_data = json.loads(json_string)
        return stop_words_data.get("roman_urdu_stop_words", [])

    def _load_chtgpt_stopwords(self, csv_path):
        """
        Load stopwords from Chtgpt's CSV file.
        
        Args:
            csv_path (str): Path to the CSV file containing stopwords.
        
        Returns:
            list: List of stopwords.
        """
        df = pd.read_csv(csv_path, header=None)  # Load CSV with no column names
        return df.iloc[:, 0].tolist()  # Extract the first column as stopwords list

    def remove_stopwords(self, tokenized_list, method='haseeb'):
        """
        Remove stopwords from a tokenized list using the specified method.
        
        Args:
            tokenized_list (list): List of tokens (words).
            method (str): Method to use for stopwords removal ('haseeb' or 'chtgpt').
        
        Returns:
            list: Tokenized list without stopwords.
        """
        if method == 'haseeb':
            if not self.haseeb_stopwords:
                raise ValueError("Haseeb Elahi stopwords not loaded. Provide a valid JSON string.")
            return [word for word in tokenized_list if word.lower() not in self.haseeb_stopwords]
        
        elif method == 'chtgpt':
            if not self.chtgpt_stopwords:
                raise ValueError("Chtgpt stopwords not loaded. Provide a valid CSV path.")
            return [word for word in tokenized_list if word.lower() not in self.chtgpt_stopwords]
        
        else:
            raise ValueError(f"Unsupported method: {method}")


class Tokenizer:
    def __init__(self, dictionary_path=None):
        """
        Initialize the Tokenizer class with optional dictionary path.
        """
        self.nlp = spacy.blank('xx')  # Blank spaCy pipeline for multi-language support
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.dictionary = self.load_dictionary(dictionary_path) if dictionary_path else None

    def load_dictionary(self, filepath):
        """
        Load a dictionary from a pickle file.
        """
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading dictionary: {e}")
            return None

    def tokenize(self, text, model_name='spacy'):
        """
        Tokenize the input text using the specified model.
        Supported models: 'bert', 'nltk', 'spacy', 'dictionary'.
        """
        if model_name == 'bert':
            return self._tokenize_bert(text)
        elif model_name == 'nltk':
            return self._tokenize_nltk(text)
        elif model_name == 'spacy':
            return self._tokenize_spacy(text)
        elif model_name == 'dictionary':
            return self._tokenize_dictionary(text)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

    def _tokenize_bert(self, text):
        """
        Tokenize using BERT tokenizer.
        """
        return self.bert_tokenizer.tokenize(text)

    def _tokenize_nltk(self, text):
        """
        Tokenize using NLTK's word_tokenize.
        """
        return word_tokenize(text)

    def _tokenize_spacy(self, text):
        """
        Tokenize using spaCy.
        """
        doc = self.nlp(text)
        return [token.text for token in doc]

    def _tokenize_dictionary(self, text):
        """
        Tokenize using a custom dictionary.
        """
        if not self.dictionary:
            raise ValueError("Dictionary not loaded. Please provide a valid dictionary path.")
        
        # Lowercase the text
        text = text.lower()

        # Remove punctuation
        punctuation = '''!()%\n٪-;۔،:\n\/'"\,“./؟_ء'''
        for char in punctuation:
            text = text.replace(char, '')

        # Split into words
        tokens = text.split()

        # Handle bigrams using the dictionary
        bi_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and f"{tokens[i]} {tokens[i + 1]}" in self.dictionary:
                bi_tokens.append(f"{tokens[i]} {tokens[i + 1]}")
                i += 2
            else:
                bi_tokens.append(tokens[i])
                i += 1

        return bi_tokens


# Main Function
if __name__ == "__main__":
    # Paths to resources
    input_csv = "input_data.csv"
    output_csv = "processed_data.csv"
    dictionary_path = "/content/dictionary.pkl"
    haseeb_json = '''
    {
        "roman_urdu_stop_words": [
            "ai", "ayi", "hy", "hai", "main", "ki", "tha", "koi", "ko", "sy", "woh", 
            "bhi", "aur", "wo", "yeh", "rha", "hota", "ho", "ga", "ka", "le", "lye", 
            "kr", "kar", "lye", "liye", "hotay", "waisay", "gya", "gaya", "kch", "ab",
            "thy", "thay", "houn", "hain", "han", "to", "is", "hi", "jo", "kya", "thi",
            "se", "pe", "phr", "wala", "waisay", "us", "na", "ny", "hun", "rha", "raha",
            "ja", "rahay", "abi", "uski", "ne", "haan", "acha", "nai", "sent", "photo", 
            "you", "kafi", "gai", "rhy", "kuch", "jata", "aye", "ya", "dono", "hoa", 
            "aese", "de", "wohi", "jati", "jb", "krta", "lg", "rahi", "hui", "karna", 
            "krna", "gi", "hova", "yehi", "jana", "jye", "chal", "mil", "tu", "hum", "par", 
            "hay", "kis", "sb", "gy", "dain", "krny", "tou"
        ]
    }
    '''
    chtgpt_csv = "chtgpt_stopwords.csv"

    # Initialize classes
    stopword_remover = StopwordRemover(haseeb_json=haseeb_json, chtgpt_csv=chtgpt_csv)
    tokenizer = Tokenizer(dictionary_path=dictionary_path)

    # Read the input CSV file
    df = pd.read_csv(input_csv)

    # Ensure the required columns exist
    if 'paragraph' not in df.columns or 'category' not in df.columns:
        raise ValueError("CSV must contain 'paragraph' and 'category' columns.")

    # Process each row
    df['tokenized_words'] = ""
    df['stopword_removed_words'] = ""

    for index, row in df.iterrows():
        paragraph = row['paragraph']
        
        # Tokenize the paragraph
        tokenized_words = tokenizer.tokenize(paragraph, model_name='spacy')
        
        # Remove stopwords
        stopword_removed_words = stopword_remover.remove_stopwords(tokenized_words, method='haseeb')
        
        # Save results to the DataFrame
        df.at[index, 'tokenized_words'] = str(tokenized_words)
        df.at[index, 'stopword_removed_words'] = str(stopword_removed_words)

    # Save the processed DataFrame to a new CSV file
    df.to_csv(output_csv, index=False)
    print(f"Processed data saved to {output_csv}")